In [1]:
from pathlib import Path
import pandas as pd

RAW = Path("../data/raw")
PROCESSED = Path("../data/processed")
PROCESSED.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(
    RAW / "quality_control_data.csv",
    sep=";",
    encoding="cp1251",
    dtype=str,
    keep_default_na=False,
)

print(df.shape)
df.head()

(42480, 10)


,trade_name,no_ru,dt_ru,series,producer,country,status,type,letter_number,letter_date
0,"Капотен, таблетки 25 мг 20 шт., упаковки ячейк...",ЛП-№(001970)-(РГ-RU),15.03.2023,2340224,"Акционерное общество ""Химико-фармацевтический ...",Россия,недоброкачественное ЛС,Отзыв,01И-1161/26,09.10.2026
1,"ИБУПРОФЕН ВЕЛФАРМ, таблетки, покрытые пленочно...",ЛП-№(004109)-(РГ-RU),21.12.2023,2520825,"Общество с ограниченной ответственностью ""Велф...",Россия,недоброкачественное ЛС,Отзыв,01И-1159/26,08.10.2026
2,"Симетикон, капсулы 40 мг 10 шт., упаковки ячей...",ЛП-008595,04.10.2022,181025,"Общество с ограниченной ответственностью ""Прои...",Россия,недоброкачественное ЛС,Отзыв,01И-1151/26,06.10.2026
3,"Бронхорус®, таблетки 30 мг 10 шт., упаковки яч...",ЛП-№(000535)-(РГ-RU),27.01.2022,91025,"Публичное акционерное общество ""Акционерное Ку...",Россия,недоброкачественное ЛС,Прекращение обращения,01И-1149/26,06.10.2026
4,"Кеторолак, раствор для внутривенного и внутрим...",ЛП-№(001499)-(РГ-RU),02.12.2022,160425,"Публичное акционерное общество ""Акционерное Ку...",Россия,недоброкачественное ЛС,Прекращение обращения,01И-1150/26,06.10.2026


In [3]:
df.columns

Index(['trade_name', 'no_ru', 'dt_ru', 'series', 'producer', 'country',
       'status', 'type', 'letter_number', 'letter_date'],
      dtype='object')

In [2]:
rename_map = {
    "trade_name": "drug_name",
    "no_ru": "registration_number",
    "dt_ru": "registration_date",
    "series": "batch_series",
    "producer": "manufacturer",
    "country": "manufacturer_country",
    "status": "quality_status",
    "type": "action_type",
    "letter_number": "letter_number",
    "letter_date": "letter_date",
}

df = df.rename(columns=rename_map)
df.columns.tolist()

['drug_name',
 'registration_number',
 'registration_date',
 'batch_series',
 'manufacturer',
 'manufacturer_country',
 'quality_status',
 'action_type',
 'letter_number',
 'letter_date']

In [3]:
df.columns

Index(['drug_name', 'registration_number', 'registration_date', 'batch_series',
       'manufacturer', 'manufacturer_country', 'quality_status', 'action_type',
       'letter_number', 'letter_date'],
      dtype='object')

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 42480 entries, 0 to 42479
Data columns (total 10 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   drug_name             42480 non-null  object
 1   registration_number   42480 non-null  object
 2   registration_date     42480 non-null  object
 3   batch_series          42480 non-null  object
 4   manufacturer          42480 non-null  object
 5   manufacturer_country  42480 non-null  object
 6   quality_status        42480 non-null  object
 7   action_type           42480 non-null  object
 8   letter_number         42480 non-null  object
 9   letter_date           42480 non-null  object
dtypes: object(10)
memory usage: 3.2+ MB


In [5]:
df.head()

,drug_name,registration_number,registration_date,batch_series,manufacturer,manufacturer_country,quality_status,action_type,letter_number,letter_date
0,"Капотен, таблетки 25 мг 20 шт., упаковки ячейк...",ЛП-№(001970)-(РГ-RU),15.03.2023,2340224,"Акционерное общество ""Химико-фармацевтический ...",Россия,недоброкачественное ЛС,Отзыв,01И-1161/26,09.10.2026
1,"ИБУПРОФЕН ВЕЛФАРМ, таблетки, покрытые пленочно...",ЛП-№(004109)-(РГ-RU),21.12.2023,2520825,"Общество с ограниченной ответственностью ""Велф...",Россия,недоброкачественное ЛС,Отзыв,01И-1159/26,08.10.2026
2,"Симетикон, капсулы 40 мг 10 шт., упаковки ячей...",ЛП-008595,04.10.2022,181025,"Общество с ограниченной ответственностью ""Прои...",Россия,недоброкачественное ЛС,Отзыв,01И-1151/26,06.10.2026
3,"Бронхорус®, таблетки 30 мг 10 шт., упаковки яч...",ЛП-№(000535)-(РГ-RU),27.01.2022,91025,"Публичное акционерное общество ""Акционерное Ку...",Россия,недоброкачественное ЛС,Прекращение обращения,01И-1149/26,06.10.2026
4,"Кеторолак, раствор для внутривенного и внутрим...",ЛП-№(001499)-(РГ-RU),02.12.2022,160425,"Публичное акционерное общество ""Акционерное Ку...",Россия,недоброкачественное ЛС,Прекращение обращения,01И-1150/26,06.10.2026


In [6]:
import re

def normalize_text(value: str) -> str:
    if not isinstance(value, str):
        return ""
    value = value.strip().lower()
    value = re.sub(r"\s+", " ", value)
    value = re.sub(r"[«»\'`®]", "", value)
    value = re.sub(r"\s*,\s*", ", ", value)
    return value

df["drug_name_clean"] = df["drug_name"].map(normalize_text)
df["manufacturer_clean"] = df["manufacturer"].map(normalize_text)
df["quality_status_clean"] = df["quality_status"].map(normalize_text)
df["action_type_clean"] = df["action_type"].map(normalize_text)
df["manufacturer_country_clean"] = df["manufacturer_country"].map(normalize_text)

In [7]:
for col in ["drug_name_clean",
    "manufacturer_clean",
    "quality_status_clean",
    "action_type_clean",
    "manufacturer_country_clean",
]:
    print("\n", col)
    print(df[col].value_counts().head(15))


 drug_name_clean
drug_name_clean
лозап плюс, таблетки, покрытые пленочной оболочкой 12.5 мг+50 мг 15 шт., блистеры (6), пачки картонные                                                                           638
кардиомагнил, таблетки покрытые пленочной оболочкой 75 мг+15.2 мг 100 шт., флаконы темного стекла,                                                                               511
кардиомагнил, таблетки покрытые пленочной оболочкой 150мг+30, 39мг 100 шт., флаконы темного стекла                                                                               433
лозап плюс, таблетки, покрытые пленочной оболочкой 12.5 мг+50 мг 15 шт., блистеры (2), пачки картонные                                                                           336
ретвисет, капсулы 100 мг 10 шт., упаковки ячейковые контурные (3), пачки картонные                                                                                               332
эналаприл, таблетки 10 мг 10 шт., упаковки ячейковые контурны

In [11]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 42480 entries, 0 to 42479
Data columns (total 15 columns):
 #   Column                      Non-Null Count  Dtype         
---  ------                      --------------  -----         
 0   drug_name                   42480 non-null  object        
 1   registration_number         42480 non-null  object        
 2   registration_date           42001 non-null  datetime64[ns]
 3   batch_series                42480 non-null  object        
 4   manufacturer                42480 non-null  object        
 5   manufacturer_country        42480 non-null  object        
 6   quality_status              42480 non-null  object        
 7   action_type                 42480 non-null  object        
 8   letter_number               42480 non-null  object        
 9   letter_date                 42480 non-null  datetime64[ns]
 10  drug_name_clean             42480 non-null  object        
 11  manufacturer_clean          42480 non-null  object    

In [9]:
date_cols = ["registration_date", "letter_date"]

for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors="coerce", dayfirst=True)

df[["registration_date", "letter_date"]].describe()

,registration_date,letter_date
count,42001,42480
mean,2008-03-01 05:59:08.058379520,2015-06-25 12:56:22.372881408
min,1964-04-30 00:00:00,2005-01-17 00:00:00
25%,2007-05-29 00:00:00,2010-06-17 00:00:00
50%,2008-12-04 00:00:00,2016-02-09 00:00:00
75%,2010-06-28 00:00:00,2020-02-27 00:00:00
max,2025-07-03 00:00:00,2026-10-09 00:00:00


In [10]:
print("letter_date:", df["letter_date"].min(), "—", df["letter_date"].max())
print("Нераспознанных дат письма:", df["letter_date"].isna().sum())
print("Нераспознанных дат регистрации:", df["registration_date"].isna().sum())

letter_date: 2005-01-17 00:00:00 — 2026-10-09 00:00:00
Нераспознанных дат письма: 0
Нераспознанных дат регистрации: 479


In [23]:
df[df["registration_date"].isna()]["registration_date"].describe()

count      0
mean     NaT
min      NaT
25%      NaT
50%      NaT
75%      NaT
max      NaT
Name: registration_date, dtype: object

In [26]:
df.head(2)

,drug_name,registration_number,registration_date,batch_series,manufacturer,manufacturer_country,quality_status,action_type,letter_number,letter_date,drug_name_clean,manufacturer_clean,quality_status_clean,action_type_clean,manufacturer_country_clean,drug_name_short,drug_name_ost
0,"Капотен, таблетки 25 мг 20 шт., упаковки ячейк...",ЛП-№(001970)-(РГ-RU),2023-03-15,2340224,"Акционерное общество ""Химико-фармацевтический ...",Россия,недоброкачественное ЛС,Отзыв,01И-1161/26,2026-10-09,"капотен, таблетки 25 мг 20 шт., упаковки ячейк...","акционерное общество ""химико-фармацевтический ...",недоброкачественное лс,отзыв,россия,капотен,"таблетки 25 мг 20 шт., упаковки ячейковые кон..."
1,"ИБУПРОФЕН ВЕЛФАРМ, таблетки, покрытые пленочно...",ЛП-№(004109)-(РГ-RU),2023-12-21,2520825,"Общество с ограниченной ответственностью ""Велф...",Россия,недоброкачественное ЛС,Отзыв,01И-1159/26,2026-10-08,"ибупрофен велфарм, таблетки, покрытые пленочно...","общество с ограниченной ответственностью ""велф...",недоброкачественное лс,отзыв,россия,ибупрофен велфарм,"таблетки, покрытые пленочной оболочкой 400 мг..."


In [25]:
# Разбиваем по запятой и создаём новые столбцы
df[['drug_name_short','drug_name_ost']] = df['drug_name_clean'].str.split(',', expand=True, n=1)

In [27]:
df["event_key"] = (
    df["letter_number"].fillna("") + "|" +
    df["letter_date"].dt.strftime("%Y-%m-%d").fillna("") + "|" +
    df["drug_name_short"].fillna("") + "|" +
    df["batch_series"].fillna("") + "|" +
    df["manufacturer_clean"].fillna("")
)

print("Всего строк:", len(df))
print("Уникальных event_key:", df["event_key"].nunique())
print("Дубликатов:", len(df) - df["event_key"].nunique())

Всего строк: 42480
Уникальных event_key: 41732
Дубликатов: 748


In [28]:
df.columns

Index(['drug_name', 'registration_number', 'registration_date', 'batch_series',
       'manufacturer', 'manufacturer_country', 'quality_status', 'action_type',
       'letter_number', 'letter_date', 'drug_name_clean', 'manufacturer_clean',
       'quality_status_clean', 'action_type_clean',
       'manufacturer_country_clean', 'drug_name_short', 'drug_name_ost',
       'event_key'],
      dtype='object')

In [29]:
df[["event_key"]]

,event_key
0,01И-1161/26|2026-10-09|капотен|2340224|акционе...
1,01И-1159/26|2026-10-08|ибупрофен велфарм|25208...
2,01И-1151/26|2026-10-06|симетикон|181025|общест...
3,01И-1149/26|2026-10-06|бронхорус|91025|публичн...
4,01И-1150/26|2026-10-06|кеторолак|160425|публич...
...,...
42475,01И-11/05|2005-01-19|пиралгин|260304|белмедпре...
42476,01И-12/05|2005-01-19|левомицетина таблетки|150...
42477,01И-12/05|2005-01-19|амиксина таблетки 0.125 г...
42478,01И-8/05|2005-01-17|валокордин|403031|кревель ...


In [31]:
df[df.duplicated("event_key", keep=False)]

,drug_name,registration_number,registration_date,batch_series,manufacturer,manufacturer_country,quality_status,action_type,letter_number,letter_date,drug_name_clean,manufacturer_clean,quality_status_clean,action_type_clean,manufacturer_country_clean,drug_name_short,drug_name_ost,event_key
33,"Фуросемид буфус, раствор для внутривенного и в...",ЛП-№(002140)-(РГ-RU),2023-04-10,21224,"АО ""Производственная фармацевтическая компания...",Россия,недоброкачественное ЛС,О снятии с посерийного контроля,01И-924/26,2026-08-11,"фуросемид буфус, раствор для внутривенного и в...","ао ""производственная фармацевтическая компания...",недоброкачественное лс,о снятии с посерийного контроля,россия,фуросемид буфус,раствор для внутривенного и внутримышечного в...,01И-924/26|2026-08-11|фуросемид буфус|21224|ао...
37,"Фуросемид буфус, раствор для внутривенного и в...",ЛП-№(002140)-(РГ-RU),2023-04-10,90424,"АО ""Производственная фармацевтическая компания...",Россия,недоброкачественное ЛС,О снятии с посерийного контроля,01И-924/26,2026-08-11,"фуросемид буфус, раствор для внутривенного и в...","ао ""производственная фармацевтическая компания...",недоброкачественное лс,о снятии с посерийного контроля,россия,фуросемид буфус,раствор для внутривенного и внутримышечного в...,01И-924/26|2026-08-11|фуросемид буфус|90424|ао...
38,"Фуросемид буфус, раствор для внутривенного и в...",ЛП-№(002140)-(РГ-RU),2023-04-10,90424,"АО ""Производственная фармацевтическая компания...",Россия,недоброкачественное ЛС,О разрешении к реализации серии,01И-924/26,2026-08-11,"фуросемид буфус, раствор для внутривенного и в...","ао ""производственная фармацевтическая компания...",недоброкачественное лс,о разрешении к реализации серии,россия,фуросемид буфус,раствор для внутривенного и внутримышечного в...,01И-924/26|2026-08-11|фуросемид буфус|90424|ао...
39,"Фуросемид буфус, раствор для внутривенного и в...",ЛП-№(002140)-(РГ-RU),2023-04-10,21224,"АО ""Производственная фармацевтическая компания...",Россия,недоброкачественное ЛС,О разрешении к реализации серии,01И-924/26,2026-08-11,"фуросемид буфус, раствор для внутривенного и в...","ао ""производственная фармацевтическая компания...",недоброкачественное лс,о разрешении к реализации серии,россия,фуросемид буфус,раствор для внутривенного и внутримышечного в...,01И-924/26|2026-08-11|фуросемид буфус|21224|ао...
41,"АМЛОДИПИН-ПРАНА, таблетки 5 мг 10 шт., упаковк...",ЛП-№(002073)-(РГ-RU),2023-03-30,470725,"Общество с ограниченной ответственностью ""ПРАН...",Россия,недоброкачественное ЛС,О разрешении к реализации серии,02И-913/26,2026-08-07,"амлодипин-прана, таблетки 5 мг 10 шт., упаковк...","общество с ограниченной ответственностью ""пран...",недоброкачественное лс,о разрешении к реализации серии,россия,амлодипин-прана,"таблетки 5 мг 10 шт., упаковки ячейковые конт...",02И-913/26|2026-08-07|амлодипин-прана|470725|о...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
39919,"Церебролизин, раствор для инъекций ~ шт., амп...",П N013827/01,2007-07-08,502698,Эбеве Фарма Гес.м.б.Х. Нфг. КГ,Австрия,сомнительное ЛС,Фальсификация приостановление,01И-216/07,2007-03-28,"церебролизин, раствор для инъекций ~ шт., ампу...",эбеве фарма гес.м.б.х. нфг. кг,сомнительное лс,фальсификация приостановление,австрия,церебролизин,"раствор для инъекций ~ шт., ампулы темного ст...",01И-216/07|2007-03-28|церебролизин|502698|эбев...
39979,"Реополиглюкин, раствор для инфузий [в растворе...",Р N001783/01-2002,2002-10-09,2540704,Биохимик ОАО,Россия,сомнительное ЛС,Фальсификация (изъятие по признакам),01И-184/07,2007-03-19,"реополиглюкин, раствор для инфузий [в растворе...",биохимик оао,сомнительное лс,фальсификация (изъятие по признакам),россия,реополиглюкин,раствор для инфузий [в растворе натрия хлорид...,01И-184/07|2007-03-19|реополиглюкин|2540704|би...
39981,"Реополиглюкин, раствор для инфузий [в растворе...",Р N001783/01-2002,2002-10-09,2540704,Биохимик ОАО,Россия,сомнительное ЛС,Фальсификация (изъятие по признакам),01И-184/07,2007-03-19,"реопол

In [32]:
duplicates = df[df.duplicated("event_key", keep=False)].sort_values("event_key")
duplicates.head(5)

,drug_name,registration_number,registration_date,batch_series,manufacturer,manufacturer_country,quality_status,action_type,letter_number,letter_date,drug_name_clean,manufacturer_clean,quality_status_clean,action_type_clean,manufacturer_country_clean,drug_name_short,drug_name_ost,event_key
17895,"Стрептокиназа, лиофилизат для приготовления ра...",П N010658,2011-09-27,011115,Республиканское унитарное производственное пре...,Республика Беларусь,недоброкачественное ЛС,Изъятие,01И-1003/17,2017-04-24,"стрептокиназа, лиофилизат для приготовления ра...",республиканское унитарное производственное пре...,недоброкачественное лс,изъятие,республика беларусь,стрептокиназа,лиофилизат для приготовления раствора для вну...,01И-1003/17|2017-04-24|стрептокиназа|011115|ре...
17894,"Стрептокиназа, лиофилизат для приготовления ра...",П N010658,2011-09-27,011115,Республиканское унитарное производственное пре...,Республика Беларусь,сомнительное ЛС,Изъятие,01И-1003/17,2017-04-24,"стрептокиназа, лиофилизат для приготовления ра...",республиканское унитарное производственное пре...,сомнительное лс,изъятие,республика беларусь,стрептокиназа,лиофилизат для приготовления раствора для вну...,01И-1003/17|2017-04-24|стрептокиназа|011115|ре...
17893,"Стрептокиназа, лиофилизат для приготовления ра...",П N010658,2011-09-27,011115,Республиканское унитарное производственное пре...,Республика Беларусь,фальсифицированное ЛС,Изъятие,01И-1003/17,2017-04-24,"стрептокиназа, лиофилизат для приготовления ра...",республиканское унитарное производственное пре...,фальсифицированное лс,изъятие,республика беларусь,стрептокиназа,лиофилизат для приготовления раствора для вну...,01И-1003/17|2017-04-24|стрептокиназа|011115|ре...
19801,"Октретекс®, раствор для инфузий и подкожного в...",ЛП-001885,2012-10-24,010315,"ООО ""Эллара""",Россия,недоброкачественное ЛС,Отзыв,01И-1004/16,2016-05-18,"октретекс, раствор для инфузий и подкожного вв...","ооо ""эллара""",недоброкачественное лс,отзыв,россия,октретекс,"раствор для инфузий и подкожного введения 0, ...","01И-1004/16|2016-05-18|октретекс|010315|ооо ""э..."
19802,"Октретекс®, раствор для инфузий и подкожного в...",ЛП-001885,2012-10-24,010315,"ООО ""Эллара""",Россия,недоброкачественное ЛС,Отзыв,01И-1004/16,2016-05-18,"октретекс, раствор для инфузий и подкожного вв...","ооо ""эллара""",недоброкачественное лс,отзыв,россия,октретекс,"раствор для инфузий и подкожного введения 0, ...","01И-1004/16|2016-05-18|октретекс|010315|ооо ""э..."


In [33]:
full_duplicates = df.duplicated().sum()
print("Полных дубликатов:", full_duplicates)

#df = df.drop_duplicates()

Полных дубликатов: 0


In [34]:
df["event_month"] = df["letter_date"].dt.to_period("M").astype(str)

df[["letter_date", "event_month"]].head()

,letter_date,event_month
0,2026-10-09,2026-10
1,2026-10-08,2026-10
2,2026-10-06,2026-10
3,2026-10-06,2026-10
4,2026-10-06,2026-10


In [29]:
#pip install pyarrow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.0/55.0 MB 6.5 MB/s  0:00:08m0:00:0100:01
Note: you may need to restart the kernel to use updated packages.


In [36]:
df.to_parquet(PROCESSED / "quality_events_clean.parquet", index=False)

In [35]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 42480 entries, 0 to 42479
Data columns (total 19 columns):
 #   Column                      Non-Null Count  Dtype         
---  ------                      --------------  -----         
 0   drug_name                   42480 non-null  object        
 1   registration_number         42480 non-null  object        
 2   registration_date           42001 non-null  datetime64[ns]
 3   batch_series                42480 non-null  object        
 4   manufacturer                42480 non-null  object        
 5   manufacturer_country        42480 non-null  object        
 6   quality_status              42480 non-null  object        
 7   action_type                 42480 non-null  object        
 8   letter_number               42480 non-null  object        
 9   letter_date                 42480 non-null  datetime64[ns]
 10  drug_name_clean             42480 non-null  object        
 11  manufacturer_clean          42480 non-null  object    

In [17]:
print(df.shape)
print(df["event_month"].min(), "—", df["event_month"].max())
print(df["manufacturer_clean"].nunique(), "уникальных производителей")
print(df["drug_name_clean"].nunique(), "уникальных препаратов")
print(df["quality_status_clean"].value_counts().head(10))
print(df["action_type_clean"].value_counts().head(10))

(42480, 17)
2005-01 — 2026-10
1900 уникальных производителей
7000 уникальных препаратов
quality_status_clean
недоброкачественное лс     26513
оригинальное лс            12908
сомнительное лс             2372
фальсифицированное лс        346
незаконный оборот            236
незарегистрированное лс      100
                               5
Name: count, dtype: int64
action_type_clean
отзыв                                15097
брак производителя (территория)       5497
приостановление реализации            4424
отзыв производителем                  2966
брак производителя россия             2915
прекращение обращения                 2221
приостановление                       2071
изъятие                                935
о переводе на посерийный контроль      720
разрешение реализации                  697
Name: count, dtype: int64


In [24]:
df.head()

,drug_name,registration_number,registration_date,batch_series,manufacturer,manufacturer_country,quality_status,action_type,letter_number,letter_date,drug_name_clean,manufacturer_clean,quality_status_clean,action_type_clean,manufacturer_country_clean,event_key,event_month
0,"Капотен, таблетки 25 мг 20 шт., упаковки ячейк...",ЛП-№(001970)-(РГ-RU),2023-03-15,2340224,"Акционерное общество ""Химико-фармацевтический ...",Россия,недоброкачественное ЛС,Отзыв,01И-1161/26,2026-10-09,"капотен, таблетки 25 мг 20 шт., упаковки ячейк...","акционерное общество ""химико-фармацевтический ...",недоброкачественное лс,отзыв,россия,"01И-1161/26|2026-10-09|капотен, таблетки 25 мг...",2026-10
1,"ИБУПРОФЕН ВЕЛФАРМ, таблетки, покрытые пленочно...",ЛП-№(004109)-(РГ-RU),2023-12-21,2520825,"Общество с ограниченной ответственностью ""Велф...",Россия,недоброкачественное ЛС,Отзыв,01И-1159/26,2026-10-08,"ибупрофен велфарм, таблетки, покрытые пленочно...","общество с ограниченной ответственностью ""велф...",недоброкачественное лс,отзыв,россия,"01И-1159/26|2026-10-08|ибупрофен велфарм, табл...",2026-10
2,"Симетикон, капсулы 40 мг 10 шт., упаковки ячей...",ЛП-008595,2022-10-04,181025,"Общество с ограниченной ответственностью ""Прои...",Россия,недоброкачественное ЛС,Отзыв,01И-1151/26,2026-10-06,"симетикон, капсулы 40 мг 10 шт., упаковки ячей...","общество с ограниченной ответственностью ""прои...",недоброкачественное лс,отзыв,россия,"01И-1151/26|2026-10-06|симетикон, капсулы 40 м...",2026-10
3,"Бронхорус®, таблетки 30 мг 10 шт., упаковки яч...",ЛП-№(000535)-(РГ-RU),2022-01-27,91025,"Публичное акционерное общество ""Акционерное Ку...",Россия,недоброкачественное ЛС,Прекращение обращения,01И-1149/26,2026-10-06,"бронхорус, таблетки 30 мг 10 шт., упаковки яче...","публичное акционерное общество ""акционерное ку...",недоброкачественное лс,прекращение обращения,россия,"01И-1149/26|2026-10-06|бронхорус, таблетки 30 ...",2026-10
4,"Кеторолак, раствор для внутривенного и внутрим...",ЛП-№(001499)-(РГ-RU),2022-12-02,160425,"Публичное акционерное общество ""Акционерное Ку...",Россия,недоброкачественное ЛС,Прекращение обращения,01И-1150/26,2026-10-06,"кеторолак, раствор для внутривенного и внутрим...","публичное акционерное общество ""акционерное ку...",недоброкачественное лс,прекращение обращения,россия,"01И-1150/26|2026-10-06|кеторолак, раствор для ...",2026-10


In [19]:
df_new['drug_name_clean']=df['drug_name_clean']

In [29]:
df.head()

,drug_name,registration_number,registration_date,batch_series,manufacturer,manufacturer_country,quality_status,action_type,letter_number,letter_date,drug_name_clean,manufacturer_clean,quality_status_clean,action_type_clean,manufacturer_country_clean,event_key,event_month,drug_name_short,drug_name_ost
0,"Капотен, таблетки 25 мг 20 шт., упаковки ячейк...",ЛП-№(001970)-(РГ-RU),2023-03-15,2340224,"Акционерное общество ""Химико-фармацевтический ...",Россия,недоброкачественное ЛС,Отзыв,01И-1161/26,2026-10-09,"капотен, таблетки 25 мг 20 шт., упаковки ячейк...","акционерное общество ""химико-фармацевтический ...",недоброкачественное лс,отзыв,россия,"01И-1161/26|2026-10-09|капотен, таблетки 25 мг...",2026-10,капотен,"таблетки 25 мг 20 шт., упаковки ячейковые кон..."
1,"ИБУПРОФЕН ВЕЛФАРМ, таблетки, покрытые пленочно...",ЛП-№(004109)-(РГ-RU),2023-12-21,2520825,"Общество с ограниченной ответственностью ""Велф...",Россия,недоброкачественное ЛС,Отзыв,01И-1159/26,2026-10-08,"ибупрофен велфарм, таблетки, покрытые пленочно...","общество с ограниченной ответственностью ""велф...",недоброкачественное лс,отзыв,россия,"01И-1159/26|2026-10-08|ибупрофен велфарм, табл...",2026-10,ибупрофен велфарм,"таблетки, покрытые пленочной оболочкой 400 мг..."
2,"Симетикон, капсулы 40 мг 10 шт., упаковки ячей...",ЛП-008595,2022-10-04,181025,"Общество с ограниченной ответственностью ""Прои...",Россия,недоброкачественное ЛС,Отзыв,01И-1151/26,2026-10-06,"симетикон, капсулы 40 мг 10 шт., упаковки ячей...","общество с ограниченной ответственностью ""прои...",недоброкачественное лс,отзыв,россия,"01И-1151/26|2026-10-06|симетикон, капсулы 40 м...",2026-10,симетикон,"капсулы 40 мг 10 шт., упаковки ячейковые конт..."
3,"Бронхорус®, таблетки 30 мг 10 шт., упаковки яч...",ЛП-№(000535)-(РГ-RU),2022-01-27,91025,"Публичное акционерное общество ""Акционерное Ку...",Россия,недоброкачественное ЛС,Прекращение обращения,01И-1149/26,2026-10-06,"бронхорус, таблетки 30 мг 10 шт., упаковки яче...","публичное акционерное общество ""акционерное ку...",недоброкачественное лс,прекращение обращения,россия,"01И-1149/26|2026-10-06|бронхорус, таблетки 30 ...",2026-10,бронхорус,"таблетки 30 мг 10 шт., упаковки ячейковые кон..."
4,"Кеторолак, раствор для внутривенного и внутрим...",ЛП-№(001499)-(РГ-RU),2022-12-02,160425,"Публичное акционерное общество ""Акционерное Ку...",Россия,недоброкачественное ЛС,Прекращение обращения,01И-1150/26,2026-10-06,"кеторолак, раствор для внутривенного и внутрим...","публичное акционерное общество ""акционерное ку...",недоброкачественное лс,прекращение обращения,россия,"01И-1150/26|2026-10-06|кеторолак, раствор для ...",2026-10,кеторолак,раствор для внутривенного и внутримышечного в...


In [31]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 42480 entries, 0 to 42479
Data columns (total 19 columns):
 #   Column                      Non-Null Count  Dtype         
---  ------                      --------------  -----         
 0   drug_name                   42480 non-null  object        
 1   registration_number         42480 non-null  object        
 2   registration_date           42001 non-null  datetime64[ns]
 3   batch_series                42480 non-null  object        
 4   manufacturer                42480 non-null  object        
 5   manufacturer_country        42480 non-null  object        
 6   quality_status              42480 non-null  object        
 7   action_type                 42480 non-null  object        
 8   letter_number               42480 non-null  object        
 9   letter_date                 42480 non-null  datetime64[ns]
 10  drug_name_clean             42480 non-null  object        
 11  manufacturer_clean          42480 non-null  object    

In [31]:
df_new.columns

RangeIndex(start=0, stop=14, step=1)

In [34]:
for c in df_new.columns:
    print(f"\nColumn name: {c}")
    print(df_new[c].value_counts().head(10))


Column name: 0
0
кардиомагнил    1196
лозап плюс      1155
валз             755
лозап            732
аммиак           689
эналаприл        565
ретвисет         553
ранитидин        518
валз н           500
эреспал          448
Name: count, dtype: int64

Column name: 1
1
таблетки                                                       4466
таблетки 250 мг 10 шт.                                          617
раствор для наружного применения и ингаляций 10% шт.            601
таблетки 500 мг 10 шт.                                          572
настойка ~ шт.                                                  545
таблетки покрытые пленочной оболочкой 75 мг+15.2 мг 100 шт.     511
таблетки 5 мг 10 шт.                                            460
таблетки покрытые пленочной оболочкой 150мг+30                  435
таблетки 50 мг 10 шт.                                           434
таблетки 10 мг 10 шт.                                           423
Name: count, dtype: int64

Column name: 2
2
упак

In [36]:
df_new[0].value_counts().head(10)

0
кардиомагнил    1196
лозап плюс      1155
валз             755
лозап            732
аммиак           689
эналаприл        565
ретвисет         553
ранитидин        518
валз н           500
эреспал          448
Name: count, dtype: int64

In [37]:
df["drug_name_clean"].value_counts().head(10)

drug_name_clean
лозап плюс, таблетки, покрытые пленочной оболочкой 12.5 мг+50 мг 15 шт., блистеры (6), пачки картонные                       638
кардиомагнил, таблетки покрытые пленочной оболочкой 75 мг+15.2 мг 100 шт., флаконы темного стекла,                           511
кардиомагнил, таблетки покрытые пленочной оболочкой 150мг+30, 39мг 100 шт., флаконы темного стекла                           433
лозап плюс, таблетки, покрытые пленочной оболочкой 12.5 мг+50 мг 15 шт., блистеры (2), пачки картонные                       336
ретвисет, капсулы 100 мг 10 шт., упаковки ячейковые контурные (3), пачки картонные                                           332
эналаприл, таблетки 10 мг 10 шт., упаковки ячейковые контурные (2), пачки картонные                                          312
эреспал, сироп 2 мг/мл 1 шт. (250.000), флаконы (1), пачки картонные                                                         253
ранитидин, таблетки покрытые пленочной оболочкой 150 мг 10 шт. (0.000), упаковки 